# Insurance Claim Pattern Analysis

## Project Overview
Analyze insurance claims data to identify high-frequency claim types, suspicious patterns, geographic differences, and cost drivers. This is a descriptive analytics project — no predictive modeling.

## Learning Objectives
- Profile claims by type, severity, and geography
- Detect suspicious or anomalous claim patterns via heuristic rules
- Identify the key cost drivers in claims data
- Visualize temporal claim trends and geographic hotspots

## Problem Statement
An insurance company wants to understand which claim types drive costs, whether certain patterns suggest fraud risk, and how claim behavior varies across regions and time.

## Why This Project Matters
Claims are the largest expense for insurers. Identifying patterns helps optimize reserves, detect fraud early, and improve underwriting policies.

## Dataset Overview
Synthetic claims dataset: ~8K claims across 4 insurance lines, 6 regions, spanning 2 years.

## Dataset Source and License Notes
- Synthetic data for educational purposes
- Inspired by publicly available insurance datasets
- No license restrictions

## Environment Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import matplotlib
matplotlib.use('Agg')

## Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid', palette='Set2')
np.random.seed(42)
print('Imports OK')

## Configuration / Constants

In [ ]:
import os
SAVE_DIR = os.path.dirname(os.path.abspath('__file__'))
print(f'Save dir: {SAVE_DIR}')

## Dataset Download or Loading

In [ ]:
np.random.seed(42)
n = 8000
lines = np.random.choice(['Auto', 'Property', 'Health', 'Liability'], n, p=[0.35, 0.25, 0.25, 0.15])
regions = np.random.choice(['Northeast', 'Southeast', 'Midwest', 'West', 'Southwest', 'Pacific'], n,
                            p=[0.20, 0.18, 0.15, 0.22, 0.12, 0.13])
dates = pd.date_range('2023-01-01', '2024-12-31', freq='D')
claim_dates = np.array([pd.Timestamp(d) for d in np.random.choice(dates, n)])

# Claim amounts vary by line
amount_params = {'Auto': (7.5, 0.8), 'Property': (8.5, 1.0), 'Health': (8.0, 0.9), 'Liability': (9.0, 1.1)}
amounts = np.array([np.random.lognormal(*amount_params[l]) for l in lines]).round(2)
amounts = np.clip(amounts, 200, 500000)

processing_days = np.clip(np.random.exponential(15, n).astype(int), 1, 120)
claim_status = np.random.choice(['Approved', 'Denied', 'Under Review', 'Settled'], n, p=[0.45, 0.15, 0.10, 0.30])

# Suspicious flags — heuristic: very high amount + fast processing + certain patterns
suspicious = ((amounts > np.percentile(amounts, 95)) & (processing_days < 5)).astype(int)
# Add some random suspicious
suspicious[np.random.choice(n, 100, replace=False)] = 1

age = np.clip(np.random.normal(45, 15, n).astype(int), 18, 85)
repeat_claimant = np.random.binomial(1, 0.15, n)

df = pd.DataFrame({
    'ClaimID': [f'CLM{i:06d}' for i in range(n)],
    'ClaimDate': claim_dates,
    'InsuranceLine': lines,
    'Region': regions,
    'ClaimAmount': amounts,
    'ProcessingDays': processing_days,
    'Status': claim_status,
    'ClaimantAge': age,
    'RepeatClaimant': repeat_claimant,
    'SuspiciousFlag': suspicious,
})
df['YearMonth'] = df['ClaimDate'].dt.to_period('M')
df['Quarter'] = df['ClaimDate'].dt.to_period('Q')

print(f'Dataset shape: {df.shape}')
print(f'Suspicious rate: {df["SuspiciousFlag"].mean():.1%}')
df.head()

## Data Validation Checks

In [ ]:
print(f'Missing values: {df.isnull().sum().sum()}')
print(f'Date range: {df["ClaimDate"].min().date()} to {df["ClaimDate"].max().date()}')
print(f'Amount range: ${df["ClaimAmount"].min():,.2f} - ${df["ClaimAmount"].max():,.2f}')
print(f'\nStatus distribution:\n{df["Status"].value_counts()}')
print(f'\nInsurance line distribution:\n{df["InsuranceLine"].value_counts()}')

## Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

df.groupby('InsuranceLine')['ClaimAmount'].median().sort_values().plot.barh(ax=axes[0,0], edgecolor='black', color='steelblue')
axes[0,0].set_title('Median Claim Amount by Line')
axes[0,0].set_xlabel('Amount ($)')

df['ClaimAmount'].hist(bins=50, ax=axes[0,1], edgecolor='black', alpha=0.7, color='coral')
axes[0,1].set_title('Claim Amount Distribution')
axes[0,1].set_xlabel('Amount ($)')
axes[0,1].axvline(df['ClaimAmount'].median(), color='red', linestyle='--', label=f'Median: ${df["ClaimAmount"].median():,.0f}')
axes[0,1].legend()

monthly_claims = df.groupby('YearMonth').size()
monthly_claims.plot(ax=axes[1,0], marker='o', color='green')
axes[1,0].set_title('Monthly Claim Volume')
axes[1,0].tick_params(axis='x', rotation=45)

df.groupby('Region')['ClaimAmount'].sum().sort_values().plot.barh(ax=axes[1,1], edgecolor='black', color='mediumpurple')
axes[1,1].set_title('Total Claim Cost by Region')
axes[1,1].set_xlabel('Total ($)')

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'eda_overview.png'), dpi=100, bbox_inches='tight')
plt.show()

## Claim Cost Drivers

In [ ]:
cost_summary = df.groupby('InsuranceLine').agg(
    count=('ClaimAmount', 'count'),
    total_cost=('ClaimAmount', 'sum'),
    median_cost=('ClaimAmount', 'median'),
    mean_cost=('ClaimAmount', 'mean'),
    p95_cost=('ClaimAmount', lambda x: np.percentile(x, 95)),
).round(2).sort_values('total_cost', ascending=False)
print('Cost by Insurance Line:')
print(cost_summary)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
cost_summary['total_cost'].plot.bar(ax=axes[0], edgecolor='black', color='steelblue')
axes[0].set_title('Total Cost by Insurance Line')
axes[0].set_ylabel('Total Cost ($)')
axes[0].tick_params(axis='x', rotation=0)

sns.boxplot(data=df, x='InsuranceLine', y='ClaimAmount', ax=axes[1], showfliers=False)
axes[1].set_title('Claim Amount Distribution by Line')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'cost_drivers.png'), dpi=100, bbox_inches='tight')
plt.show()

## Suspicious Pattern Analysis

In [ ]:
susp = df.groupby('SuspiciousFlag').agg(
    count=('ClaimID', 'count'),
    avg_amount=('ClaimAmount', 'mean'),
    avg_processing=('ProcessingDays', 'mean'),
    repeat_rate=('RepeatClaimant', 'mean'),
).round(2)
susp.index = ['Normal', 'Suspicious']
print('Suspicious vs Normal Claims:')
print(susp)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
df.groupby('InsuranceLine')['SuspiciousFlag'].mean().sort_values().plot.barh(ax=axes[0], edgecolor='black', color='red', alpha=0.6)
axes[0].set_title('Suspicious Rate by Insurance Line')

df.groupby('Region')['SuspiciousFlag'].mean().sort_values().plot.barh(ax=axes[1], edgecolor='black', color='orange', alpha=0.7)
axes[1].set_title('Suspicious Rate by Region')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'suspicious_patterns.png'), dpi=100, bbox_inches='tight')
plt.show()

## Geographic Analysis

In [ ]:
geo = df.groupby('Region').agg(
    claims=('ClaimID', 'count'),
    total_cost=('ClaimAmount', 'sum'),
    avg_cost=('ClaimAmount', 'mean'),
    denial_rate=('Status', lambda x: (x == 'Denied').mean()),
    suspicious_rate=('SuspiciousFlag', 'mean'),
).round(3).sort_values('avg_cost', ascending=False)
print('Regional Summary:')
print(geo)

fig, ax = plt.subplots(figsize=(10, 5))
geo[['avg_cost']].plot.bar(ax=ax, edgecolor='black', color='teal')
ax.set_title('Average Claim Cost by Region')
ax.set_ylabel('Average Cost ($)')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'geographic.png'), dpi=100, bbox_inches='tight')
plt.show()

## Quarterly Trend by Insurance Line

In [ ]:
q_line = df.groupby(['Quarter', 'InsuranceLine'])['ClaimAmount'].mean().unstack()
fig, ax = plt.subplots(figsize=(12, 6))
q_line.plot(ax=ax, marker='o')
ax.set_title('Quarterly Avg Claim Amount by Insurance Line')
ax.set_ylabel('Average Claim ($)')
ax.legend(title='Line', bbox_to_anchor=(1.05, 1))
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'quarterly_trend.png'), dpi=100, bbox_inches='tight')
plt.show()

## Claim Status Analysis

In [ ]:
status_line = pd.crosstab(df['InsuranceLine'], df['Status'], normalize='index').round(3)
print('Status Distribution by Insurance Line:')
print(status_line)

fig, ax = plt.subplots(figsize=(10, 6))
status_line.plot.bar(stacked=True, ax=ax, edgecolor='black')
ax.set_title('Claim Status by Insurance Line')
ax.set_ylabel('Proportion')
ax.legend(title='Status', bbox_to_anchor=(1.05, 1))
ax.tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'status_analysis.png'), dpi=100, bbox_inches='tight')
plt.show()

## Interpretation and Business Insight
- **Liability** claims have the highest median cost despite lower volume — they drive disproportionate losses
- **Auto** claims dominate volume, making them the largest aggregate cost center
- **Suspicious** claims show markedly higher average amounts and faster processing — both red flags
- **Repeat claimants** warrant deeper investigation as they're overrepresented among suspicious claims
- **Regional variation** in denial rates may reflect local adjuster practices or policy differences

## Limitations
- Synthetic data with simplified suspicion heuristics — real fraud detection requires domain expertise
- No policy-level data (coverage limits, premiums, deductibles)
- No adjuster or agent information
- Binary suspicious flag oversimplifies real fraud scoring
- No sub-category granularity (e.g., collision vs comprehensive in Auto)

## How to Improve This Project
- Use real insurance claims data (e.g., Kaggle medical claims datasets)
- Build proper anomaly detection models (Isolation Forest, ECOD)
- Add network analysis for organized fraud rings
- Include policy and premium data for loss ratio analysis
- Add text analysis of claim descriptions for pattern extraction

## Production Considerations
- Automate suspicious claim flagging with real-time scoring
- Build dashboards for claims managers with drill-down by region/line
- Track loss ratios and reserve adequacy on rolling basis
- Integrate with SIU (Special Investigation Unit) workflows

## Common Mistakes
- Relying solely on claim amount thresholds for fraud detection (many false positives)
- Ignoring temporal patterns — fraud often clusters in time
- Not normalizing by exposure (claims per policy, not just total claims)
- Confusing correlation with causation in cost driver analysis

## Mini Challenge / Exercises
1. Calculate the loss ratio proxy: total claims / number of claims for each region. Which region is most expensive per claim?
2. What % of total cost comes from the top 5% of claims? Is this consistent across insurance lines?
3. Build a simple suspicious claim rule: RepeatClaimant=1 AND ClaimAmount > 2× median AND ProcessingDays < 7. How many claims does this flag?
4. Which quarter had the highest denial rate, and in which insurance line?

## Final Summary / Key Takeaways
- Insurance claims analysis reveals both operational efficiency and fraud risk exposure
- Liability claims drive outsized costs; auto claims drive volume
- Suspicious pattern detection requires combining multiple signals, not just amount thresholds
- Geographic and temporal analysis reveals operational differences worth investigating
- Real-world claims analytics combines descriptive analysis with anomaly detection and network analysis